# Funktionszeiger

Manchmal muss Code so flexibel sein, dass verschiedene Prozeduren oder Funktionen austauschbar sind. Funktionszeiger helfen dabei.

Ein Funktionszeiger ist nicht die Funktion selbst, sondern die Speicheradresse dieser Funktion.
Um darauf zuzugreifen, ruft man die Prozedur ohne Klammern oder Argumente auf:

In [1]:
def EvaluateSum(SummandA, SummandB):
    return SummandA + SummandB

print('3+4=', EvaluateSum(3, 4))   
print('Address of this procedure in memory is: ', EvaluateSum)


3+4= 7
Address of this procedure in memory is:  <function EvaluateSum at 0x0000017914C58540>


Ein einfaches Beispiel für Funktionszeiger folgt. Die Klasse CSignalFlowBlock verwaltet ihre eigene Übertragungsfunktion, die durch einen Funktionszeiger beschrieben wird. Zusätzlich organisiert das Signalflussblock die Weitergabe der Ausgabe an den nächsten Verarbeitungsschritt und merkt sich das letzte Ergebnis.

In [2]:
class CSignalFlowBlock(object):
    
    def __init__(self, TransferFunctionPointer):
        self.__TransferFunctionPointer = TransferFunctionPointer
        self.__Output = None
        self.__LastOutput = None
        
    def RegisterOutput(self, NewOutput):
        self.__Output = NewOutput
        
    def ProcessNewData(self, NewData):
        self.__LastOutput = self.__TransferFunctionPointer(NewData)
        if self.__Output is not None:
            self.__Output.ProcessNewData(self.__LastOutput)
        return self.__LastOutput
    
    def GetLastOutput(self):
        return self.__LastOutput

Im Folgenden werden einige einfache Übertragungsfunktionen definiert:

In [3]:
import numpy as np

def Transferfunction1(x):
    return 3*x+1

def Transferfunction2(x):
    return x**2

def Transferfunction3(x):
    if np.sum(x) > 0:
        return 1.0
    else:
        return -1.0

Mit diesem Konzept lassen sich Verarbeitungsschritte leicht verwalten:

Eine Liste von Signalflussblöcken wird erstellt. Alle Eingangsdaten gehen zum ersten Block. Jeder Block organisiert seine Übertragungsfunktion und die Weitergabe an den nächsten Block.

Das Endergebnis wird aus dem letzten Block der Liste entnommen.

In [4]:
ListOfSignalFlowBlocks = []
ListOfSignalFlowBlocks.append(CSignalFlowBlock(Transferfunction1))
ListOfSignalFlowBlocks.append(CSignalFlowBlock(Transferfunction2))
ListOfSignalFlowBlocks.append(CSignalFlowBlock(Transferfunction3))

for n in range(len(ListOfSignalFlowBlocks)-1):
    ListOfSignalFlowBlocks[n].RegisterOutput(ListOfSignalFlowBlocks[n+1])
    
ListOfSignalFlowBlocks[0].ProcessNewData(42)
print('the result of this chain of signal flow blocks is: ', ListOfSignalFlowBlocks[-1].GetLastOutput())

the result of this chain of signal flow blocks is:  1.0


## Programmieraufgabe:
Implementiere eine Übertragungsfunktion, die das neutrale Element darstellt (Eingang gleich Ausgang).